# 문제 6

[kaggle 형] train_prob.csv로 failure 예측하는 모델을 만들고, 

test_prob.csv의 각각의 입력에 대한 failure가 1일 확률 예측하여 다음과 같은 형식의 answer6.csv를 만들어라. 

측정 지표는 AUC(area under of ROC curve)이다. id 는 테스트 케이스의 id 이고, failure에는 failure가 1이 될 확률이다.

id,failure

16115, 0.1

16116, 0.2


**강사: 멀티캠퍼스 강선구(sunku0316.kang@multicampus.com, sun9sun9@gmail.com)**

In [1]:
# 실행 환경 확인

import pandas as pd
import numpy as np
import sklearn
import scipy
import statsmodels
import mlxtend
import sys
import xgboost as xgb

print(sys.version)
for i in [pd, np, sklearn, scipy, mlxtend, statsmodels, xgb]:
    print(i.__name__, i.__version__)

3.7.4 (tags/v3.7.4:e09359112e, Jul  8 2019, 20:34:20) [MSC v.1916 64 bit (AMD64)]
pandas 0.25.1
numpy 1.18.5
sklearn 0.21.3
scipy 1.5.2
mlxtend 0.15.0.0
statsmodels 0.11.1
xgboost 0.80


In [6]:
df_train = pd.read_csv('train_prob.csv', index_col=['id'])
df_test = pd.read_csv('test_prob.csv', index_col=['id'])
s_kaggle_ans = pd.read_csv('test_prob_ans.csv', index_col=['id']).iloc[:, 0]
df_train.shape, df_test.shape, s_kaggle_ans.shape

((21458, 25), (5112, 24), (5112,))

In [7]:
# From 문제 1
df_train = df_train.assign(
    na_1 = lambda x: x['measurement_3'].isna(),
    na_2 = lambda x: x['measurement_5'].isna(),
)
df_test = df_test.assign(
    na_1 = lambda x: x['measurement_3'].isna(),
    na_2 = lambda x: x['measurement_5'].isna(),
)

In [8]:
df_train['product_code'].value_counts()

C    5765
E    5343
B    5250
A    5100
Name: product_code, dtype: int64

In [9]:
df_test['product_code'].value_counts()

D    5112
Name: product_code, dtype: int64

In [11]:
# 방법 1: groupby ~ apply ~ fit_transform

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer #, random_state=123
from sklearn.linear_model import LinearRegression

X_imp = ['measurement_{}'.format(i) for i in range(3, 10)] + ['measurement_17']

imp = IterativeImputer(
    LinearRegression(fit_intercept=True), random_state=123
)

df_train[X_imp] = df_train.groupby('product_code')[X_imp].apply(
    lambda x: pd.DataFrame(imp.fit_transform(x), index=x.index, columns=x.columns)
)
df_test[X_imp] = df_test.groupby('product_code')[X_imp].apply(
    lambda x: pd.DataFrame(imp.fit_transform(x), index=x.index, columns=x.columns)
)

X_mean = ['measurement_{}'.format(i) for i in range(10, 17)]

df_train[X_mean] = df_train.groupby('product_code')[X_mean].transform(
    lambda x: x.fillna(x.mean())
)
df_test[X_mean] = df_test.groupby('product_code')[X_mean].transform(
    lambda x: x.fillna(x.mean())
)

In [13]:
# 문제 2: loading의 log 변환
# attribute_0, attribute_1 ...  : 무의미
m = pd.concat([df_train['loading'], df_test['loading']]).mean()

df_train['loading'] = df_train['loading'].fillna(m)
df_test['loading'] = df_test['loading'].fillna(m)
df_train['loading_log'] = np.log(df_train['loading'])
df_test['loading_log'] = np.log(df_test['loading'])

In [ ]:
# 문제 3: SFS 의 Feature
# 문제 4: LDA predict, transform
# 문제 4: PCA n_components = 7 
# 문제 5: RF: {'max_depth': 7, 'min_samples_split': 512, 'n_estimators': 15},